# Tool Calling

In [1]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [2]:
@function_tool
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

In [3]:
@function_tool
def get_evs_concept(concept: str) -> str:
    """
    Look up a short explanation of a common Environmental Studies concept.

    Args:
        concept: Concept name, such as "photosynthesis" or "greenhouse effect"

    Returns:
        A brief explanation of the concept.

    Raises:
        ValueError: If the concept is not one of the supported topics.
    """
    concepts = {
        "photosynthesis": (
            "Photosynthesis is the process by which green plants use sunlight, "
            "water, and carbon dioxide to make glucose and release oxygen."
        ),
        "ozone depletion": (
            "Ozone depletion is the thinning of the ozone layer, mainly caused "
            "by chemicals such as CFCs. This lets more harmful ultraviolet "
            "radiation reach Earth's surface."
        ),
        "global warming": (
            "Global warming is the long-term rise in Earth's average temperature, "
            "primarily caused by human activities that increase greenhouse gases "
            "in the atmosphere."
        ),
        "greenhouse effect": (
            "The greenhouse effect is the warming that occurs when gases in "
            "Earth's atmosphere trap some of the heat radiated from the surface."
        ),
        "topsoildegradation":( "THIS IS TO TEST MY TOOL"

        ),
    }

    key = "".join(concept.lower().replace("-", " ").split())
    normalized_concepts = {
        "".join(name.lower().split()): explanation
        for name, explanation in concepts.items()
    }

    if key not in normalized_concepts:
        supported = ", ".join(concepts)
        raise ValueError(
            f"Unknown EVS concept: {concept!r}. Supported concepts: {supported}."
        )

    return normalized_concepts[key]

In [4]:
@function_tool
def add_numbers(a: float, b: float) -> float:
    """Return the sum of two numbers."""
    return a + b

In [5]:
import math

@function_tool
def cylinder_surface_area(radius: float, height: float) -> float:
    """Return the total surface area of a cylinder: 2πr(r + h)."""
    if radius < 0 or height < 0:
        raise ValueError("Radius and height must be non-negative.")

    return 2 * math.pi * radius * (radius + height)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [ ]:
get_food_calories('banana')

In [6]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    )

In [7]:
teaching_agent = Agent(
    name = "Teaching Assistant",
    instructions = """
    You answer questions about Environmental Studies.
    For every concept question, call get_evs_concept and return its result verbatim. Do not answer from your own knowledge.
    You give concise answers.
    """,
    tools=[get_evs_concept],
    model_settings=ModelSettings(tool_choice="required", parallel_tool_calls=False),
)

In [8]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

About 185 calories (banana ~105 + apple ~80 for medium sizes).


In [9]:
with trace("EVS Assistant with tools"):
    result = await Runner.run(
        teaching_agent, "What is topsoildegradation?"
    )
    print(result.final_output)

THIS IS TO TEST MY TOOL


Enforce tools use:

In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)